In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from typing import List
import esm




In [6]:
from aguirre.config import Config
cfg = Config()
print(cfg)

Config(esm_model_name='esm2_t6_8M_UR50D', embed_dim=320, pep_max_len=42, device='mps')


In [15]:
class ESMWrapper(nn.Module):
    """Wrapper um Denoiser: Aufbau eines Embeddingraumes und decoding in Aminosäurelänge
    ### Args:
        - device (string, wird im Geräte unabhängigen Code definiert.), "cpu" als default.
    """
    def __init__(self):
        super().__init__()
        
        #==Fehler bei falschen Modus auffangen==
        mode = cfg.esm_model_name
        assert mode in ("esm2_t33_650M_UR50D", "esm2_t6_8M_UR50D"), f"Unbekannter Modus '{mode}', erlaubt sind 'esm2_t33_650M_UR50D' und 'esm2_t6_8M_UR50D'"

        #==Auswahl Moden==
        if mode == "esm2_t33_650M_UR50D":
            model,alphabet = esm.pretrained.esm2_t33_650M_UR50D() # Tupel ESM2-Model und Tokentabelle
        elif mode == "esm2_t6_8M_UR50D":
            model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
        model.eval()
        model.requires_grad_(False)
        batch_converter = alphabet.get_batch_converter()
        AS = "LAGVSERTIDPKQNFYMHWC"
        AS_list = torch.tensor([alphabet.tok_to_idx[a] for a in AS], dtype= torch.long)

        #==Attribute Instanzen==
        self.register_buffer("AS_list", AS_list)
        self.AS = AS
        self.model = model
        self.alphabet = alphabet
        self.batch_converter = batch_converter
        self.device = cfg.device
        self.L_max = cfg.pep_max_len
        self.to(self.device)
        

    @torch.no_grad() #Protokoll für Backprop ausschalten
    def encode(self, seqs: List[str]) -> tuple[torch.Tensor, list[int]]: 
        """
        ### Encoder.
            Baut aus einem Batch von Peptidsequenzen ein ESM-2 Embedding. BOS und EOS werden abgeschnitten, 
            sodass Ausgabe mit Shape (B,L_batch, D) ausgegeben wird. 

        ### Args:
        - seqs als Inputliste mit Sequenzen, z.B ["ACDEFGHIKLMN", "KLVFFAED"]
        """

        #==Fehler bei L_batch > L_max auffangen==
        max_len = max(len(s)for s in seqs)
        assert max_len  <= self.L_max, f"Sequenz mit {max_len} Residuen, L_max ist {self.L_max}"
        data = [(f"seq{i}", s) for i, s in enumerate(seqs)]
        seqs_lengths = torch.tensor([len(s) for s in seqs]).unsqueeze(1).to(self.device) #dims (B,1) jeweilige AS-Sequenzlängen
        model = self.model

        #==ESM-2 Inferenz==
        _, _,seq_tokens = self.batch_converter(data) # dims: (B,L_batch+2), mit PADs zum Auffüllen
        out = model(seq_tokens.to(self.device), repr_layers = [model.num_layers]) #dims: (B)
        reps = out["representations"][model.num_layers] # dims: (B,L_batch, D)
        reps_sliced = reps[:,1:-1,:] #Abschneiden von BOS und EOS

        #==Zero-Padding mit F.pad()==
        zero_pad_amount = self.L_max - reps_sliced.size(1) # dims:(B)
        reps_padded = F.pad(reps_sliced,(0,0,0,zero_pad_amount), mode= "constant", value = 0).to(self.device)

        #==Paddingmaske mit booleans==
        positions = torch.arange(self.L_max).unsqueeze(0).to(self.device) #dims: (1, L_max)
        padding_mask = positions < seqs_lengths        

        #==Ausgabe==
        reps_return = reps_padded * padding_mask.unsqueeze(-1) # Aufgefüllt mit batch_converter Padding und 0
        return reps_return.cpu(), seqs_lengths.squeeze(1).cpu().tolist() # dims: representations (B,L_max, dims), Sequenzlängen (B, 1)

    @torch.no_grad()
    def decode(self, reps: torch.Tensor, lengths: List[int], names: List[str] = None) -> dict:

        names = names or [f"Sequenz {i+1}" for i in range(reps.size(0))]
        logits = self.model.lm_head(reps.to(self.device)) # lm_head: Linear -> GELU -> LayerNorm -> Linear. dims: (B, L_batch, 33)
        seq_tokens = torch.argmax(logits[:,:, self.AS_list], dim=-1) # Nur die Tokens für die 20 kanonischen AS
        
        results = {}
        for name, row, l in zip(names, seq_tokens.cpu().tolist(), lengths):
            seq = "" #leerer string
            for i in row[:l]:
                seq += self.AS[i]
            results[name] = seq

        return results

        #input dims: B,L_max, dims und vielleicht eine Padding-Maske? (kann eine KI eine Paddingmaske mitlernen?)
        


In [16]:
ESM = ESMWrapper()
seqs = ["GIGKFLHSAKKFGKAFVGEIMNS", "GIGKFLHSAKKFGKAFVGEIMNS"]
reps, lengths = ESM.encode(seqs)
print(ESM.decode(reps, lengths))   

{'Sequenz 1': 'MIGKFLHSAKKFGKAFVGEIMNS', 'Sequenz 2': 'MIGKFLHSAKKFGKAFVGEIMNS'}


In [ ]:
x = torch.randint(low=0, high=10, size=(2, 10,3))



design_len = 5
B, T = x.size(0), x.size(1)
            
# Create a tensor of zeros with size [B, T]
padding_mask = torch.zeros((B, T), dtype=torch.bool, device = x.device)
            
# Set values to one after the design_len+1 position, but leave the last position as one
padding_mask[:, design_len+1:] = 1

padding_mask

In [ ]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()

pep_a = "ACDEFGHIKLMNRADFR"   
pep_b = "KLVFFAED"
list = [pep_a, pep_b]
data = [(f"seq{i}", s) for i, s in enumerate(list)]
seqs_lengths = torch.tensor([len(s) for s in list]).unsqueeze(1)

positions = torch.arange(42).unsqueeze(0)
padding_mask = positions < seqs_lengths


_, _,seq_tokens = batch_converter(data) # dims (B,L_batch), mit PADs zum Auffüllen
out = model(seq_tokens.to("cpu"), repr_layers = [model.num_layers]) #dims (B)
reps = out["representations"][model.num_layers]
reps_sliced = reps[:,1:-1,:]

reps_B = reps_sliced.shape[0]
reps_dims = reps_sliced.shape[2]
reps_len = reps_sliced.shape[1]
residues = 42 - reps_len
resiues_tensor = torch.zeros(size=(reps_B, residues, reps_dims)).to("cpu")
reps_return = torch.cat(tensors=(reps_sliced, resiues_tensor), dim= 1) # Aufgefüllt mit batch_converter Padding und 


print(padding_mask.shape, reps_return.shape)
reps_return = reps_return * padding_mask.unsqueeze(-1)

reps_return[0,:, 0]
